In [16]:
#Визуализация политик, функции ценности состояний, Q функции ценности действий
#Оценка и оптимизация политик
#Расчет вероятности успеха и среднего вознаграждения при выбранной политике

In [17]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import random

np.set_printoptions(suppress=True, precision=5)
random.seed(123)
np.random.seed(123)

In [18]:
#MDP - Марковский процесс принятия решения
#Стохастическая прогулка
#50.01% - успех действия (движемся в выбранном направлении), 33,33% - остаёмся в том же состоянии,
#16.66% - движемся в противоположном направлении

P = {
    0: {
        0: [(1.0, 0, 0.0, True)],
        1: [(1.0, 0, 0.0, True)]
    },
    1: {
        0: [(0.5001, 0, 0.0, True),
            (0.3333, 1, 0.0, False),
            (0.1666, 2, 0.0, False)
        ],
        1: [(0.5001, 2, 0.0, False),
            (0.3333, 1, 0.0, False),
            (0.1666, 0, 0.0, True)
        ]
    },
    2: {
        0: [(0.5001, 1, 0.0, False),
            (0.3333, 2, 0.0, False),
            (0.1666, 3, 0.0, False)
        ],
        1: [(0.5001, 3, 0.0, False),
            (0.3333, 2, 0.0, False),
            (0.1666, 1, 0.0, False)
        ]
    },
    3: {
        0: [(0.5001, 2, 0.0, False),
            (0.3333, 3, 0.0, False),
            (0.1666, 4, 0.0, False)
        ],
        1: [(0.5001, 4, 0.0, False),
            (0.3333, 3, 0.0, False),
            (0.1666, 2, 0.0, False)
        ]
    },
    4: {
        0: [(0.5001, 3, 0.0, False),
            (0.3333, 4, 0.0, False),
            (0.1666, 5, 0.0, False)
        ],
        1: [(0.5001, 5, 0.0, False),
            (0.3333, 4, 0.0, False),
            (0.1666, 3, 0.0, False)
        ]
    },
    5: {
        0: [(0.5001, 4, 0.0, False),
            (0.3333, 5, 0.0, False),
            (0.1666, 6, 1.0, True)
        ],
        1: [(0.5001, 6, 1.0, True),
            (0.3333, 5, 0.0, False),
            (0.1666, 4, 0.0, False)
        ]
    },
    6: {
        0: [(1.0, 6, 0.0, True)],
        1: [(1.0, 6, 0.0, True)]
    }
}

In [19]:
# Визуализация политики
def print_policy(pi, P, action_symbols=('<', 'v', '>', '^'), n_cols=4, title='Выбранная политика'):
    print('Название политики =',title)
    arrs = {k:v for k,v in enumerate(action_symbols)}
    for s in range(len(P)):
        a = pi(s)
        print("| ", end="")
        if np.all([done for action in P[s].values() for _, _, _, done in action]):
            #Терминальные состояния
            print("".rjust(9), end=" ")
        else:
            # Нетерминальные состояния
            print(str(s).zfill(2), arrs[a].rjust(6), end=" ")
        if (s + 1) % n_cols == 0: print("|")

In [20]:
# Визуализация функции оценки состояния
def print_state_value_function(V, P, n_cols=4, prec=3, title='Функция оценки состояния:'):
    print(title)
    for s in range(len(P)):
        v = V[s]
        print("| ", end="")
        if np.all([done for action in P[s].values() for _, _, _, done in action]):
            print("".rjust(9), end=" ")
        else:
            print(str(s).zfill(2), '{}'.format(np.round(v, prec)).rjust(6), end=" ")
        if (s + 1) % n_cols == 0: print("|")

In [21]:
#from pprint import pprint
#from tqdm import tqdm_notebook as tqdm

from tabulate import tabulate
import itertools

#Визуализация функции ценности действий
def print_action_value_function(Q,
                                optimal_Q=None,
                                action_symbols=('<', '>'),
                                prec=3,
                                title='Функция оценки ценности действий:'):
    vf_types=('',) if optimal_Q is None else ('', '*', 'err')
    headers = ['s',] + [' '.join(i) for i in list(itertools.product(vf_types, action_symbols))]
    print(title)
    states = np.arange(len(Q))[..., np.newaxis]
    arr = np.hstack((states, np.round(Q, prec)))
    if not (optimal_Q is None):
        arr = np.hstack((arr, np.round(optimal_Q, prec), np.round(optimal_Q-Q, prec)))
    print(tabulate(arr, headers, tablefmt="fancy_grid"))

In [22]:
# Начальное состояние
init_state = 3
# Целевое состояние с наградой +1
goal_state = 7

LEFT, RIGHT = range(2)
#LEFT =0, RIGHT = 1

#Политика - движение влево на каждом состоянии
pi = lambda s: { 0:LEFT, 1:LEFT, 2:LEFT, 3:LEFT, 4:LEFT, 5:LEFT, 6:LEFT }[s]
print_policy(pi, P, action_symbols=('<', '>'), n_cols=6)

Название политики = Выбранная политика
|           | 01      < | 02      < | 03      < | 04      < | 05      < |
|           

In [23]:
# Расчет ценности состояний
# pi - политика
# P - MDP
# gamma - коэффициент дисконтирования
# theta - эпсилон, для анализа сходимости

def policy_evaluation(pi, P, gamma=1.0, theta=1e-7):
    prev_V = np.zeros(len(P), dtype=np.float64)
    counter = 0
    while True:
        counter=counter+1
        V = np.zeros(len(P), dtype=np.float64)
        for s in range(len(P)):
            for prob, next_state, reward, done in P[s][pi(s)]:
                V[s] += prob * (reward + gamma * prev_V[next_state] * (not done))
        if np.max(np.abs(prev_V - V)) < theta:
            break
        prev_V = V.copy()
    return V

In [24]:
V = policy_evaluation(pi, P)

In [25]:
print_state_value_function(V, P, n_cols=7, prec=5, title='Функция ценности состояния:')

Функция ценности состояния:
|           | 01 0.00274 | 02 0.01096 | 03 0.03565 | 04 0.10976 | 05 0.33222 |           |


In [26]:
#Алгоритм оптимизации политик
# Расчет функции оценки действий Q
# Выбор действия с максимальной ценностью
def policy_improvement(V, P, gamma=1.0):
    Q = np.zeros((len(P), len(P[0])), dtype=np.float64)
    for s in range(len(P)):
        for a in range(len(P[s])):
            for prob, next_state, reward, done in P[s][a]:
                Q[s][a] += prob * (reward + gamma * V[next_state] * (not done))
    print('Значения функции ценности действий Q = \n',Q)
    new_pi = lambda s: {s:a for s, a in enumerate(np.argmax(Q, axis=1))}[s]
    return new_pi,Q

In [27]:
new_pi,Q = policy_improvement(V,P)

Значения функции ценности действий Q = 
 [[0.      0.     ]
 [0.00274 0.0064 ]
 [0.01096 0.02194]
 [0.03565 0.0686 ]
 [0.10976 0.20867]
 [0.33222 0.62912]
 [0.      0.     ]]


In [28]:
print_policy(new_pi, P, action_symbols=('<', '>'), n_cols=6)

Название политики = Выбранная политика
|           | 01      > | 02      > | 03      > | 04      > | 05      > |
|           

In [29]:
print_action_value_function(Q, optimal_Q=None,
                                action_symbols=('<', '>'),
                                prec=3,
                                title='Функция оценки ценности действий:')

Функция оценки ценности действий:
╒═════╤═══════╤═══════╕
│   s │     < │     > │
╞═════╪═══════╪═══════╡
│   0 │ 0     │ 0     │
├─────┼───────┼───────┤
│   1 │ 0.003 │ 0.006 │
├─────┼───────┼───────┤
│   2 │ 0.011 │ 0.022 │
├─────┼───────┼───────┤
│   3 │ 0.036 │ 0.069 │
├─────┼───────┼───────┤
│   4 │ 0.11  │ 0.209 │
├─────┼───────┼───────┤
│   5 │ 0.332 │ 0.629 │
├─────┼───────┼───────┤
│   6 │ 0     │ 0     │
╘═════╧═══════╧═══════╛
